In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

^C
   ---------------------------------------- 0.0/39.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/39.1 MB ? eta -:--:--
   ----- ---------------------------------- 5.8/39.1 MB 29.6 MB/s eta 0:00:02
   ------------- -------------------------- 13.1/39.1 MB 32.0 MB/s eta 0:00:01
   --------------------- ------------------ 20.7/39.1 MB 33.7 MB/s eta 0:00:01
   ---------------------------- ----------- 28.3/39.1 MB 34.0 MB/s eta 0:00:01
   ------------------------------------- -- 36.7/39.1 MB 34.8 MB/s eta 0:00:01
   ---------------------------------------- 39.1/39.1 MB 32.4 MB/s  0:00:01



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Cloning https://github.com/unslothai/unsloth.git to C:\Users\sompy\AppData\Local\Temp\pip-install-ljmcdcfc\unsloth_ac64ac434ef64a72853ae5609d1d08ee
  Resolved https://github.com/unslothai/unsloth.git to commit faca4e2278f381a402404fc54186da9cd053e517
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached bitsandbytes-0.50.2-py3-none-win_amd64.whl.metadata (11 kB)
   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   ---------------------------------------- 529.0/529.0 kB 6.5 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ----------------------- ---------------- 7.3/12.3 MB 33.1 MB/s eta 0:00:01
   ---------------------------------------- 12

  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git 'C:\Users\sompy\AppData\Local\Temp\pip-install-ljmcdcfc\unsloth_ac64ac434ef64a72853ae5609d1d08ee'

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detect (Float16 for T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth supports 0 for maximum speed
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves 30% VRAM
    random_state = 3407,
)

In [ ]:
import json
from datasets import Dataset

system_prompt = (
    "You are BharatFinanceEdu, an expert Indian financial educator. "
    "Your goal is to explain financial concepts simply, using clear analogies, "
    "practical examples, and dispelling common misconceptions. "
    "Always respond in the language asked."
)

formatted_data = []
skipped_lines = 0

with open("final_train.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if not line.strip():
            continue

        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            skipped_lines += 1
            continue # Safely skips the smashed line causing your error

        # Build structured response body
        answer_parts = [f"**Explanation:** {item.get('simple_answer', '')}"]
        if item.get("analogy"):
            answer_parts.append(f"**Analogy:** {item['analogy']}")
        if item.get("example"):
            answer_parts.append(f"**Example:** {item['example']}")
        if item.get("misconception"):
            answer_parts.append(f"**Common Misconception:** {item['misconception']}")

        full_assistant_reply = "\n\n".join(answer_parts)

        # Apply standard conversational format
        conversation = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": item.get("question", "")},
            {"role": "assistant", "content": full_assistant_reply}
        ]

        try:
            formatted_text = tokenizer.apply_chat_template(
                conversation,
                tokenize=False,
                add_generation_prompt=False
            )
            formatted_data.append({"text": formatted_text})
        except Exception:
            skipped_lines += 1
            continue

dataset = Dataset.from_list(formatted_data)
print(f"Successfully loaded and formatted {len(dataset)} training examples.")
print(f"Skipped {skipped_lines} corrupted lines.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 150, # Fast 15-minute run for the hackathon
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# This line actually starts the GPU training
trainer_stats = trainer.train()

In [ ]:
model.save_pretrained_merged("bharat_finance_model", tokenizer, save_method="lora")
!zip -r bharat_finance_model.zip bharat_finance_model

In [ ]:
# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

def test_model(question):
    # We use the exact same system prompt from the training step
    messages = [
        {"role": "system", "content": (
            "You are BharatFinanceEdu, an expert Indian financial educator. "
            "Your goal is to explain financial concepts simply, using clear analogies, "
            "practical examples, and dispelling common misconceptions. "
            "Always respond in the language asked."
        )},
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=1024,
        temperature=0.3, # Low temperature keeps financial answers grounded
        use_cache=True
    )

    # Decode only the newly generated tokens
    response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]

    print(f"Question: {question}")
    print("-" * 40)
    print(response)
    print("=" * 60 + "\n")

# Test 1: Core financial concept (English)
test_model("What is compound interest and why is it important?")

# Test 2: Common investment vehicle (Hindi)
test_model("चक्रवृद्धि ब्याज (Compound Interest) क्या है और यह क्यों महत्वपूर्ण है?")

# Test 3: Market basics (Marathi)
test_model("चक्रवाढ व्याज (Compound Interest) म्हणजे काय आणि ते का महत्त्वाचे आहे?")

In [ ]:
def generate_quiz(topic, num_questions=5):
    # A new, strict system prompt forcing a quiz format
    quiz_system_prompt = (
        "You are a strict financial quiz master. Your task is to generate a multiple-choice "
        "quiz on the requested topic. Do NOT use the Explanation/Analogy format. "
        "Output exactly in this format:\n"
        "1. [Question]\n"
        "A) [Option]\nB) [Option]\nC) [Option]\nD) [Option]\n"
        "Answer: [Correct Option]"
    )

    messages = [
        {"role": "system", "content": quiz_system_prompt},
        {"role": "user", "content": f"Generate a {num_questions}-question beginner quiz on {topic}."}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # Slightly higher temperature (0.5) to allow for creative question generation
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=600,
        temperature=0.5,
        use_cache=True
    )

    response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]

    print(f"--- {topic.upper()} QUIZ ---")
    print(response)
    print("=" * 50)

# Test it out!
generate_quiz("Mutual Funds and Diversification", num_questions=5)
generate_quiz("शेअर बाजार (Stock Market in Marathi)", num_questions=3)

In [ ]:
# 1. Install yfinance inside Colab
!pip install -q yfinance

import yfinance as yf
from unsloth import FastLanguageModel

# Ensure the model is in fast inference mode
FastLanguageModel.for_inference(model)

# 2. Live stock helper function
def get_live_stock_data(ticker_symbol):
    try:
        if not ticker_symbol.endswith(".NS"):
            ticker_symbol += ".NS"
        stock = yf.Ticker(ticker_symbol)
        info = stock.info
        price = info.get("currentPrice", "N/A")
        beta = info.get("beta", "N/A")
        high_52 = info.get("fiftyTwoWeekHigh", "N/A")
        low_52 = info.get("fiftyTwoWeekLow", "N/A")

        if beta != "N/A":
            risk_level = (
                "High Volatility / High Risk" if float(beta) > 1.2 else
                "Moderate Risk (Moves with index)" if float(beta) > 0.8 else
                "Low Volatility / Conservative Risk"
            )
        else:
            risk_level = "Risk profile unavailable"

        return (
            f"[LIVE MARKET DATA - {ticker_symbol}]: "
            f"Current Price: ₹{price} | 52-Week High: ₹{high_52} | 52-Week Low: ₹{low_52} | "
            f"Volatility Metric (Beta {beta}): {risk_level}."
        )
    except Exception as e:
        return ""

# 3. Test Function simulating RAG + Live Stock + Model
def run_evaluation(user_question, simulated_rag_context="", ticker=""):
    print(f"\nEvaluating Question: '{user_question}'")

    live_data = get_live_stock_data(ticker) if ticker else ""
    if live_data:
        print(f"-> Fetched API: {live_data}")

    system_instruction = (
        "You are BharatFinanceEdu, an expert Indian financial educator. "
        "Explain financial concepts simply, using clear analogies, "
        "practical examples, and dispelling common misconceptions. "
        "Use any provided document context and live market data. "
        "Always respond in the language asked."
    )

    full_context = f"Official SEBI/RBI Guidelines:\n{simulated_rag_context}\n\n{live_data}"

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": f"{full_context}\n\nUser Question: {user_question}"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=350,
        temperature=0.3,
        use_cache=True
    )

    response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
    print("\n" + "="*50)
    print(response)
    print("="*50 + "\n")

# --- TEST 1: Stock Question with Live API Data ---
run_evaluation(
    user_question="Is Tesla a risky investment right now, what is the risk rate?",
    simulated_rag_context="SEBI advises retail investors to assess risk tolerance, diversify, and not invest based solely on short-term price momentum.",
    ticker="RELIANCE.NS"
)

# --- TEST 2: Hindi Question ---
run_evaluation(
    user_question="म्यूचुअल फंड में एसआईपी (SIP) शुरू करने का क्या फायदा है?",
    simulated_rag_context="AMFI guidelines explain that Systematic Investment Plans average out market volatility through Rupee Cost Averaging."
)